**Machine Learning On Berkovich Spaces For WORDNET**
Assume that we have words from WORDNET embedded in the Berkovich line $\mathcal B^1$ over $\mathbb Q_p$ in a way that respects semantic relations. So, for example, we could take $p=3$ and work with Mammals, and possibly our embedding is such that all Carnivores are mapped to Berkovich points which are 0 mod 3 (so are part of the tree under $B(0,1/3)$), and all Non-Carnivores are mapped to points which are 1 mod 3. We want to learn a function $f$ which distinguishes Carnivores from Non-Carnivores.

We guess that we can take $f$ of the form $$f_t(x) = \frac{|t_2x^2+t_1x+t_0|}{|s_2x^2+s_1x+s_0|}$$ for parameters $t=(t_0, t_1, t_2, s_0, s_1, s_2) \in \mathcal B^6$. We would like $f_t$ to be near 0 at a Carnivore, and near $\infty$ at a Non-Carnivore.

We want to minimise the following loss function:
$$\ell(t) = \sum_X \left(1 - e^{-f_t(x)} \right) + \sum_{Y} e^{-f_t(x)}$$
where $X$ is a training data set of Carnivores, and $Y$ of Non-Carnivores.

In [1]:
include("wordnet.jl")

┌ Warning: attempting to remove probably stale pidfile
│   path = /home/workspace/.julia/registries/.pid
└ @ FileWatching.Pidfile /home/workspace/.julia/juliaup/julia-1.10.0+0.x64.linux.gnu/share/julia/stdlib/v1.10/FileWatching/src/pidfile.jl:273
    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
    Updating `~/.julia/environments/v1.10/Project.toml`
⌅ [2edaba10] + Nemo v0.39.1
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`


greedy_descent (generic function with 1 method)

The field $\mathbb Q_3$ can be defined through Nemo up to some precision.

In [2]:
prec = 20
K = PadicField(3,prec)

Field of 3-adic numbers

Berkovich points can be defined by specifying a centre in $K=\mathbb Q_3$ and a radius. In order to avoid unnecessary rounding we describe the radius using the valuation instead, which is stored as an integer. We imagine the set $X$ below consists of Carnivores, which are 0 mod 3, and the set $Y$ below are Non-Carnivores, which are 1 mod 3.

In [3]:
X1 = BerkovichPoint(K(0), 4)
X2 = BerkovichPoint(K(9), 3)
X3 = BerkovichPoint(K(18), 6)
X4 = BerkovichPoint(K(0), 5)
X5 = BerkovichPoint(K(27), 2)
X = [X1, X2, X3, X4, X5]

Y1 = BerkovichPoint(K(1), 3)
Y2 = BerkovichPoint(K(10), 4)
Y3 = BerkovichPoint(K(10), 5)
Y4 = BerkovichPoint(K(19), 2)
Y = [Y1, Y2, Y3, Y4]

4-element Vector{BerkovichPoint}:
 BerkovichPoint(3^0 + O(3^20), 3)
 BerkovichPoint(3^0 + 3^2 + O(3^20), 4)
 BerkovichPoint(3^0 + 3^2 + O(3^20), 5)
 BerkovichPoint(3^0 + 2*3^2 + O(3^20), 2)

The descent algorithm "simul_descent" aims to find parameters $t \in \mathcal B^6$ that minimise the loss function $\ell(t)$. We initialise at $B(0,1)^6 \in \mathcal B^6$. At each step we search all $(p+1)^6$ possible branches that increase the (valuation of the) radius by 0 or 1. Our new parameters become the choice which minimises $\ell(t)$. In particular the radii can become assymetric. The algorithm terminates if we reach a parameter $t$ for which no other branch decreases the loss function. It also terminates if the valuation of one of the radii exceeds some bound $N$ that we may choose.

In [4]:
param = simul_descent(X,Y,5)

6-element Vector{BerkovichPoint}:
 BerkovichPoint(O(3^20), 5)
 BerkovichPoint(3^2 + O(3^20), 3)
 BerkovichPoint(3^0 + O(3^20), 1)
 BerkovichPoint(2*3^0 + 2*3^1 + 3^2 + 2*3^3 + O(3^20), 4)
 BerkovichPoint(3^0 + O(3^20), 4)
 BerkovichPoint(O(3^20), 4)

The algorithm terminates because one radius reaches $N=5$.

In [5]:
param = simul_descent(X,Y,20)

6-element Vector{BerkovichPoint}:
 BerkovichPoint(3^7 + O(3^20), 8)
 BerkovichPoint(3^2 + 2*3^3 + 2*3^4 + O(3^20), 6)
 BerkovichPoint(3^0 + O(3^20), 4)
 BerkovichPoint(2*3^0 + 2*3^1 + 3^2 + 2*3^3 + O(3^20), 4)
 BerkovichPoint(3^0 + O(3^20), 4)
 BerkovichPoint(O(3^20), 4)

With these parameters, the descent step does not return new parameters which decrease the loss function. 

In [6]:
t0, t1, t2, s0, s1, s2 = param[1], param[2], param[3], param[4], param[5], param[6]

(BerkovichPoint(3^7 + O(3^20), 8), BerkovichPoint(3^2 + 2*3^3 + 2*3^4 + O(3^20), 6), BerkovichPoint(3^0 + O(3^20), 4), BerkovichPoint(2*3^0 + 2*3^1 + 3^2 + 2*3^3 + O(3^20), 4), BerkovichPoint(3^0 + O(3^20), 4), BerkovichPoint(O(3^20), 4))

Now we use these parameters to define the function $f$, called "test_function". We can evaluate this at various choices of $x \in \mathcal B$.

In [7]:
x1 = BerkovichPoint(K(-6), 4) # 0 mod 3 example
x2 = BerkovichPoint(K(19), 3) # 1 mod 3 example
x3 = BerkovichPoint(K(35), 6) # 2 mod 3 example
x4 = BerkovichPoint(K(2), 2)
println(test_function(t0,t1,t2,s0,s1,s2, x1))
println(test_function(t0,t1,t2,s0,s1,s2, x2))
println(test_function(t0,t1,t2,s0,s1,s2, x3))
println(test_function(t0,t1,t2,s0,s1,s2, x4))

0.11111111111111109
9.000000000000002
1.0
1.0


On our original data:

In [8]:
println(test_function(t0,t1,t2,s0,s1,s2, X1))
println(test_function(t0,t1,t2,s0,s1,s2, X2))
println(test_function(t0,t1,t2,s0,s1,s2, X3))
println(test_function(t0,t1,t2,s0,s1,s2, X4))
println(test_function(t0,t1,t2,s0,s1,s2, X5))

println(test_function(t0,t1,t2,s0,s1,s2, Y1))
println(test_function(t0,t1,t2,s0,s1,s2, Y2))
println(test_function(t0,t1,t2,s0,s1,s2, Y3))
println(test_function(t0,t1,t2,s0,s1,s2, Y4))

0.0013717421124828531
0.012345679012345675
0.00015241579027587248
0.00045724737082761756
0.012345679012345675
9.000000000000002
81.00000000000003
81.00000000000003
9.000000000000002


It seems like this "test_function" can distinguish 0, 1 and 2 mod 3 by giving values <<1, >>1 and ~1 respectively. On the training data it can distinguish Carnivores quite well, but Non-Carnivores not so much. To fix this we probably need to tweak our loss function - by incorporating this exponential, the algorithm prioritises minimising the numerator of $f$ over the denominator. For example, $exp(-9) = 0.0001...$ doesn't contribute much to the loss function, but $1-exp(-1/9)= 0.105...$ does.

Allowing for more parameters would also improve our "test_function". This would involve optimising over $\mathcal B^n$ with $n > 6$, which could become computationally expensive. Instead of employing this simultaneous descent step which checks $(p+1)^n$ branches each time, we could instead randomly pick one parameter and check its $p+1$ branches, and repeat. This is greedier than the simultaneous descent but is much faster in $n$. An implementation of this is given by "greedy_descent". It terminates when a radius reaches $N$ (taken to be 20 below), or when $M$ consecutive iterations does not change our parameters (taken to be 1000 below).

In [9]:
param2 = greedy_descent(X, Y, 20, 1000)
t0, t1, t2, s0, s1, s2 = param2[1], param2[2], param2[3], param2[4], param2[5], param2[6]

(BerkovichPoint(O(3^20), 9), BerkovichPoint(O(3^20), 4), BerkovichPoint(O(3^20), 0), BerkovichPoint(O(3^20), 0), BerkovichPoint(O(3^20), 0), BerkovichPoint(O(3^20), 0))

In [10]:
println(test_function(t0,t1,t2,s0,s1,s2, X1))
println(test_function(t0,t1,t2,s0,s1,s2, X2))
println(test_function(t0,t1,t2,s0,s1,s2, X3))
println(test_function(t0,t1,t2,s0,s1,s2, X4))
println(test_function(t0,t1,t2,s0,s1,s2, X5))

println(test_function(t0,t1,t2,s0,s1,s2, Y1))
println(test_function(t0,t1,t2,s0,s1,s2, Y2))
println(test_function(t0,t1,t2,s0,s1,s2, Y3))
println(test_function(t0,t1,t2,s0,s1,s2, Y4))

0.00015241579027587248
0.012345679012345675
0.012345679012345675
5.080526342529085e-5
0.012345679012345675
1.0
1.0
1.0
1.0


Some comments on the greedy algorithm: since we are picking random parameters to optimise, we should get different outputs, but it seems that when $M$ is large enough we always obtain the same parameters. The resulting test function seems to do well at distinguishing the Carnivores but not the Non-Carnivores - again, maybe we should change the loss function. Looking at just a few iterations of the greedy_descent_step it seems that we quickly reach a point in the Berkovich space where our loss function is constant, hence why the algorithm terminates.